In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
COURSE_FOLDER = REPO_ROOT / "raw_data" / "training_courses"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run 01_load_ecsf_to_database.ipynb first"

In [ ]:
from cscaf.loading import TrainingLoader
from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
storage = Storage.of(study, read_only=False)
loader = TrainingLoader(study, storage)
courses = TrainingLoader.read(COURSE_FOLDER)
print("database:", storage.path.relative_to(REPO_ROOT))
print("courses :", len(courses), "files in", COURSE_FOLDER.relative_to(REPO_ROOT))

In [ ]:
{"rf_section": storage.count_rows("rf_section"),
 "industry_training": storage.count_rows("industry_training"),
 "industry_training_role": storage.count_rows("industry_training_role")}

In [ ]:
report = loader.load(COURSE_FOLDER)
print(f"loaded {len(report.loaded)} rows, skipped {len(report.skipped)} already present\n")
print(f"{'code':8s} {'variant':26s} {'chars':>6s}  sections taken")
for row in report.loaded:
    print(f"{row['code']:8s} {row['variant']:26s} {row['characters']:6d}  "
          f"{' | '.join(s[:38] for s in row['sections_taken'])}")
if report.skipped:
    print(f"\nskipped: {report.skipped}")
for change in report.roles_replaced:
    print(f"roles replaced: {change['code']} {change['before']} -> {change['after']}")

In [ ]:
not_english = loader.english_problems()
for code, variant, line, language, content in not_english:
    print(f"ERROR not in English: {code}/{variant} line {line} reads as {language}: {content[:110]}")
print(f"{len(not_english)} lines not in English")

In [ ]:
variants = sum(len(course["variants"]) for course in courses)
rows = storage.count_rows("industry_training")
print({"courses": len(courses), "rows expected": variants, "industry_training": rows,
       "industry_training_role": storage.count_rows("industry_training_role")})
assert rows == variants == 44, rows